# Flight Delay Prediction - API + ngrok Demo

Este notebook demuestra el empaquetado operativo del modelo de regresion y su publicacion temporal con ngrok desde Google Drive.

Objetivo de la demo:

- Montar `MyDrive/flight-delay-prediction`.
- Verificar que exista `models/flight_delay_model.joblib`.
- Verificar que exista `src/flight_delay_api.py`.
- Levantar la API FastAPI desde `src/flight_delay_api.py`.
- Publicar la API local con ngrok.
- Probar el endpoint `/health` y `/predict` desde una URL publica.

Antes de ejecutar este notebook, corre primero el notebook principal `Flight Delay Prediction EDA ML.ipynb` hasta la celda de empaquetado del modelo de regresion.

## Guion de Presentacion - API y ngrok

### 1. Objetivo del notebook

Este notebook demuestra que el modelo ya no vive solamente en el notebook de entrenamiento. Aqui cargo el modelo empaquetado, levanto una API con FastAPI, publico esa API con ngrok y hago una prediccion desde una URL publica.

### 2. Dependencias

Primero preparo las librerias necesarias para servir el modelo: FastAPI para crear el backend, Uvicorn para levantar el servidor, pyngrok para publicar el servicio y requests para probar los endpoints.

### 3. Configuracion de Drive

Aqui monto Google Drive y dejo como raiz del proyecto la carpeta flight-delay-prediction. Esto es importante porque la API necesita encontrar dos cosas: el modelo empaquetado en models/flight_delay_model.joblib y el codigo fuente en src/.

### 4. Validacion del artefacto

En esta corrida se encontro correctamente el modelo en /content/drive/MyDrive/flight-delay-prediction/models/flight_delay_model.joblib y tambien la API en /content/drive/MyDrive/flight-delay-prediction/src/flight_delay_api.py. Esto confirma que el modelo fue empaquetado desde el notebook anterior.

### 5. Levantar FastAPI

Despues levanto el backend localmente en Colab. El log muestra Application startup complete y la prueba contra /docs regreso codigo 200. Eso confirma que FastAPI esta corriendo correctamente antes de exponerlo al exterior.

### 6. Publicacion con ngrok

Luego uso ngrok para convertir el servidor local de Colab en una URL publica. Esta parte es la que permite que otra persona consuma el modelo sin tener acceso directo al notebook ni al archivo joblib.

### 7. Endpoint publico

La corrida genero una URL publica de ngrok y mostro dos rutas relevantes: /health para revisar el estado del servicio y /predict para mandar features y recibir la prediccion del modelo.

### 8. Payload de prueba

El payload incluye exactamente las features que el modelo espera: hora, temporada, fin de semana, senales operativas, promedio historico por ruta, promedio historico por aerolinea, distancia y variables de demora. Esto demuestra que la API respeta el mismo contrato de features usado durante entrenamiento.

### 9. Resultado de prediccion

La llamada a /predict regreso codigo 200, lo cual confirma que la API recibio la peticion y ejecuto el modelo. El resultado fue Random Forest Regressor, con una prediccion de 25.51 minutos de retraso de llegada y nivel de riesgo medium.

### 10. Interpretacion de negocio

La salida no es una clase ni una probabilidad; es una estimacion numerica de minutos. Como 25.51 minutos supera 15 minutos pero no llega a 45, la API lo traduce como riesgo medio. En negocio eso significa monitorear el vuelo y preparar acciones preventivas.

### Cierre de la demo

Con este notebook cierro el flujo completo: el modelo se entrena y empaqueta en el notebook anterior, se carga en una API, se publica con ngrok y se consume desde una URL publica. Esto demuestra una instancia funcional del modelo fuera del notebook de entrenamiento.

<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de dependencias del backend

Esta celda instala las librerías necesarias para servir el modelo. `fastapi` define la API, `uvicorn` levanta el servidor, `pyngrok` publica el puerto local, `requests` permite probar endpoints, `joblib` carga el modelo empaquetado y `xgboost`/`scikit-learn` son necesarias para deserializar modelos entrenados.

Aunque en este notebook no entrenamos, sí necesitamos las librerías del modelo para poder cargar el `.joblib`. Si falta alguna dependencia, la API puede fallar al iniciar o al hacer predicciones.

Frase para decir: “Esta celda prepara el ambiente de despliegue, no el ambiente de entrenamiento”.

In [1]:
import subprocess
import sys


for package in ["fastapi", "uvicorn", "pyngrok", "requests", "joblib", "pandas", "scikit-learn", "xgboost"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("Dependencias listas")

Dependencias listas


<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de configuración del proyecto

Esta celda monta Google Drive y posiciona el notebook dentro de la carpeta `flight-delay-prediction`. Esto es necesario porque el modelo y los archivos Python viven en Drive, no dentro del runtime temporal de Colab.

También se agrega la ruta del proyecto a `sys.path`. Esto permite importar módulos como `src.flight_delay_api` desde el notebook.

El resultado imprime la carpeta activa y confirma que el proyecto espera encontrar `src/`, `models/` y `data/`. Esa estructura conecta el entrenamiento con el despliegue.

Frase para decir: “Aquí configuro el entorno para que Colab encuentre el modelo y el código de API en Drive”.

In [2]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/flight-delay-prediction')
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Proyecto activo en: {PROJECT_ROOT}')
print(f'Contenido esperado: src/, models/, data/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Proyecto activo en: /content/drive/MyDrive/flight-delay-prediction
Contenido esperado: src/, models/, data/


<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de la sección de validación

Esta sección existe para comprobar que el despliegue tiene los insumos necesarios. Un backend de ML no puede funcionar sólo con el notebook; necesita el modelo guardado y el código que sabe cómo cargarlo.

En una arquitectura real, esta validación sería equivalente a revisar que el artefacto del modelo y el servicio estén listos antes de publicar una versión.

Frase para decir: “Aquí verifico que el modelo ya fue empaquetado y que la API tiene los archivos necesarios para funcionar”.

## 1. Validar artefacto del modelo

El archivo `models/flight_delay_model.joblib` debe existir. Ese archivo se genera desde el notebook principal despues de entrenar XGBoost.

<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de validación del artefacto

Esta celda valida que existan los archivos mínimos para poder servir el modelo: el `.joblib`, el archivo de API y el archivo de features.

En la corrida se encontró correctamente `flight_delay_model.joblib` en la carpeta `models/`, y también se encontró `flight_delay_api.py` en `src/`. Esto confirma que el notebook anterior sí generó el modelo empaquetado y que el código de servicio está disponible.

Esta validación evita errores silenciosos. Si falta el modelo o falta la API, el notebook se detiene antes de intentar abrir ngrok.

Frase para decir: “Antes de levantar el servicio, confirmo que el modelo empaquetado y el código de API existen”.

In [3]:
MODEL_PATH = PROJECT_ROOT / "models" / "flight_delay_model.joblib"
API_PATH = PROJECT_ROOT / "src" / "flight_delay_api.py"
FEATURES_PATH = PROJECT_ROOT / "src" / "flight_delay_features.py"

missing_paths = [path for path in [MODEL_PATH, API_PATH, FEATURES_PATH] if not path.exists()]

if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        "Faltan archivos necesarios en Google Drive:\n"
        f"{missing_text}\n\n"
        "Verifica que subiste la carpeta src/ y que ejecutaste el empaquetado del modelo."
    )

print(f"Modelo encontrado: {MODEL_PATH.resolve()}")
print(f"API encontrada: {API_PATH.resolve()}")

Modelo encontrado: /content/drive/MyDrive/flight-delay-prediction/models/flight_delay_model.joblib
API encontrada: /content/drive/MyDrive/flight-delay-prediction/src/flight_delay_api.py


<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de la sección FastAPI

Esta sección empieza el despliegue del modelo. Hasta este punto sólo validamos archivos; ahora vamos a ejecutar un servidor que pueda recibir peticiones HTTP.

La diferencia con el notebook de entrenamiento es importante: aquí no se vuelve a entrenar el modelo. La API sólo carga el `.joblib` y usa el modelo ya entrenado para responder.

Frase para decir: “En esta sección pasamos de artefacto guardado a servicio ejecutándose”.

## 2. Levantar FastAPI localmente

La API queda escuchando en `http://127.0.0.1:8000`. Se ejecuta en un thread para poder seguir usando el notebook.

<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de levantar FastAPI

Esta celda inicia el backend usando Uvicorn. FastAPI define la aplicación y Uvicorn la ejecuta como servidor HTTP en el puerto 8000.

Se ejecuta en un thread para que el notebook no se quede bloqueado. Esto permite levantar el servidor y luego seguir ejecutando celdas para probar `/docs`, abrir ngrok y hacer predicciones.

La prueba contra `/docs` devuelve código 200 y el log muestra `Application startup complete`. Eso confirma que el servidor está vivo antes de exponerlo públicamente.

Frase para decir: “Aquí levanto el backend que va a cargar el modelo empaquetado y responder predicciones”.

In [4]:
import threading
import time
import src.flight_delay_api as api
import requests
import uvicorn

def run_api():
    uvicorn.run("src.flight_delay_api:app", host="0.0.0.0", port=8000, log_level="info")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

time.sleep(5)

health_response = requests.get("http://127.0.0.1:8000/docs", timeout=30)
print(health_response.status_code)

print("FastAPI levantado correctamente")



INFO:     Started server process [9238]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:38730 - "GET /docs HTTP/1.1" 200 OK
200
FastAPI levantado correctamente


<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de la sección ngrok

Esta sección marca el paso de backend local a endpoint público. Hasta este punto, FastAPI vive dentro del runtime de Colab; con ngrok, ese backend se vuelve accesible mediante una URL externa.

La utilidad de esta parte es demostrar integración. No basta con tener un modelo entrenado: necesitamos una forma de invocarlo desde fuera, como si fuera usado por otro notebook, una app o un sistema de operaciones.

Frase para decir: “Esta es la transición entre tener un modelo local y tener un servicio consumible desde internet”.

## 3. Publicar con ngrok

Si tu cuenta de ngrok requiere token, define `NGROK_AUTHTOKEN` antes de abrir el tunel.

<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de publicación con ngrok

Esta celda abre un túnel público con ngrok. En términos simples, ngrok toma el servidor que está corriendo localmente en Colab y lo expone mediante una URL accesible desde internet.

Esto es exactamente lo que pidió el profesor cuando mencionó que se debía ver funcionando ngrok y el código donde se invoca. Aquí se ve la llamada a `ngrok.connect(8000, "http")`, que conecta el puerto local de FastAPI con una URL pública.

La salida muestra tres cosas importantes: la URL pública, el endpoint de salud y el endpoint de predicción. El endpoint clave para negocio es `/predict`, porque ahí se consume el modelo.

Frase para decir: “Aquí convierto mi API local de Colab en un servicio público temporal usando ngrok”.

In [5]:
import os

from pyngrok import ngrok

NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "360JdI2VbbFFSbl08uEhtv6a94T_3jLjgajj3f1EsW8cfk4uL")

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

ngrok.kill()
public_url = ngrok.connect(8000, "http").public_url

print("URL publica de la API:")
print(public_url)
print("Health check:", f"{public_url}/health")
print("Prediction endpoint:", f"{public_url}/predict")

URL publica de la API:
https://uncoordinate-josh-intuitively.ngrok-free.dev
Health check: https://uncoordinate-josh-intuitively.ngrok-free.dev/health
Prediction endpoint: https://uncoordinate-josh-intuitively.ngrok-free.dev/predict


<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación del payload de prueba

Esta sección prepara una solicitud de ejemplo para el endpoint `/predict`. El payload contiene un diccionario `features` con las mismas variables que el modelo espera desde entrenamiento.

Las variables incluyen hora del día, si es hora pico, si es fin de semana, temporada alta, señal de retraso acumulado, promedio histórico por ruta, promedio histórico por aerolínea, distancia y causas operativas de demora.

La importancia de esta celda es que muestra el contrato de inferencia. La API no recibe cualquier dato: espera exactamente las features con las que el modelo fue entrenado. Esto mantiene consistencia entre entrenamiento y producción.

Frase para decir: “Aquí demuestro qué información tendría que mandar un sistema externo para obtener una predicción”.

## 4. Probar prediccion desde ngrok

Este payload simula un vuelo con senales de riesgo operativo. El consumidor externo debe usar la misma estructura.

<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Interpretación de la predicción desde ngrok

Esta celda es la prueba final de la demo. Se envía un payload al endpoint público `/predict`, no al servidor local directamente. Eso significa que la petición está entrando por ngrok y llegando a la API publicada.

El resultado fue código 200, lo cual confirma que la petición fue válida y que el backend pudo ejecutar el modelo correctamente. La respuesta indica que el modelo usado fue `Random Forest Regressor`.

La predicción fue `25.51` minutos de retraso de llegada. Como ese valor está por encima de 15 minutos, pero por debajo de 45, la API lo clasifica operativamente como `medium`. Es importante decir que esta clasificación de riesgo no es el modelo; el modelo sigue siendo regresión y entrega minutos.

Frase para decir: “Esta celda prueba que alguien externo puede mandar features a una URL pública y recibir una predicción del modelo empaquetado”.

In [7]:
sample_payload = {
    "features": {
        "HOUR_OF_DAY": 17,
        "IS_PEAK_HOUR": 1,
        "IS_WEEKEND": 0,
        "IS_HIGH_SEASON": 1,
        "CASCADING_DELAY_FLAG": 1,
        "ROUTE_AVG_DELAY": 12.4,
        "AIRLINE_AVG_DELAY": 9.8,
        "AIRLINE_ENC": 3,
        "DISTANCE": 740,
        "DEPARTURE_DELAY": 18,
        "MONTH": 7,
        "DAY_OF_WEEK": 5,
        "AIR_SYSTEM_DELAY": 10,
        "WEATHER_DELAY": 0,
    }
}

prediction_response = requests.post(f"{public_url}/predict", json=sample_payload, timeout=120)
print(prediction_response.status_code)
print(prediction_response.json())

INFO:     35.232.84.198:0 - "POST /predict HTTP/1.1" 200 OK
200
{'model_name': 'Random Forest Regressor', 'predicted_arrival_delay_minutes': 25.51, 'risk_level': 'medium', 'features_used': ['HOUR_OF_DAY', 'IS_PEAK_HOUR', 'IS_WEEKEND', 'IS_HIGH_SEASON', 'CASCADING_DELAY_FLAG', 'ROUTE_AVG_DELAY', 'AIRLINE_AVG_DELAY', 'AIRLINE_ENC', 'DISTANCE', 'DEPARTURE_DELAY', 'MONTH', 'DAY_OF_WEEK', 'AIR_SYSTEM_DELAY', 'WEATHER_DELAY']}


<!-- INTERPRETACION_FLIGHT_DELAY_API_SENIOR -->
### Conclusión final de la demo API + ngrok

Este notebook demuestra que el proyecto no termina en entrenamiento. El modelo se empaquetó, se cargó desde un backend, se expuso con una URL pública y respondió una predicción real.

La parte más importante para defender ante el profesor es que el consumo del modelo ocurre fuera del notebook de EDA. Eso simula cómo otra aplicación, otro notebook o un usuario externo podría invocar el modelo mediante HTTP.

La respuesta final fue exitosa: código 200, modelo `Random Forest Regressor`, predicción de 25.51 minutos y riesgo medio. Eso completa el flujo solicitado: modelo, empaquetado, ngrok e invocación.

Frase para cerrar: “Con esto demuestro un ciclo completo de Data Science aplicado: entrenamiento, empaquetado, servicio y consumo externo”.